# Site selection

Before I can measure how well a drone detects individual trees, I need a mission where the drone's coverage overlaps a plot that a field crew already walked and measured by hand. The field data is the ground truth; without it there's nothing to check the drone against.

This notebook searches the Open Forest Observatory drone catalog for candidate missions, checks which ones contain a stem-mapped field plot, and picks the site used for the rest of this analysis.

## What's already downloaded

`src/fetch_raw_data.sh` has pulled everything in `data/raw/` that doesn't depend on which site I choose. See `data/raw/README.md` for full provenance. The short version:

| Path | Use here |
|---|---|
| `ofo_drone_catalog/stac_items_ofo.geojson` | 299 STAC items with asset URLs for orthomosaic, CHM, DSM, point cloud |
| `ofo_drone_catalog/all-mission-polygons-w-metadata.gpkg` | Official mission footprints — the **complete** catalog |
| `ofo_ground_ref_catalog/plot_centroids_derived.csv` | 296 field plots as `plot_id, lat, lon` (WGS84) |
| `ofo_ground_ref_catalog/ground-plot-catalog-datatable.html` | Plot attributes: area, tree count, species mix, min DBH, year, license |

**Two things to keep in mind while I work through this.**

1. The STAC collection is not the whole catalog. It indexes 299 missions; CyVerse hosts more. So a STAC-only search will silently drop candidates — which is exactly how Emerald Point disappears. I'll search STAC first because that's the skill worth having, then cross-check against the mission polygons file.
2. The field stem data isn't published yet — OFO's plot pages say "Coming soon." What I have is plot *metadata* and *locations*, enough to rank candidate sites, but not enough to score detections in Task 1.5. That needs an email to OFO.

Rasters stay in their native projected CRS (OFO mostly uses EPSG:3310). Vectors get reprojected to match. EPSG:4326 is only for the web map in Task 1.8.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from pystac_client import Client

STAC_URL = "https://stac.cyverse.org"
COLLECTION = "Open Forest Observatory"

RAW = Path.cwd().parent / "data" / "raw"
DRONE = RAW / "ofo_drone_catalog"
GROUND = RAW / "ofo_ground_ref_catalog"

assert DRONE.exists() and GROUND.exists(), f"expected data under {RAW}"

## 1. Open the catalog and look at one item

A STAC **Item** is one mission. It carries a footprint geometry, a datetime, a bag of
properties, and an **assets** dict — the actual downloadable files, each with an `href`,
a media type, and roles. Before searching for many, open one and see its shape.

In [ ]:
# TODO(human): open the catalog and pull a single item to inspect.
#
#   1. Client.open(STAC_URL) -> a Client
#   2. .get_collection(COLLECTION) -> the OFO collection
#   3. take one item from .get_items() (it's a generator — don't materialise all 299)
#   4. print item.id, item.datetime, item.bbox
#   5. print sorted(item.assets) and, for one asset, its .href and .media_type
#
# Question to answer in the next markdown cell: which asset keys would Task 1.4 need,
# and which would Task 1.6 need?


*(notes on what the assets are)*

## 2. Every mission as a GeoDataFrame

Careful with one thing: `/search` on this server returns empty for bbox queries. Paging the
collection's items works. The local `stac_items_ofo.geojson` is already that paging done —
use it if the API is slow, but write the live version first so the method is yours.

In [ ]:
# TODO(human): build a GeoDataFrame of mission footprints.
#
#   - collect each item's id, datetime, geometry, and whether it has an ortho and a CHM
#   - gpd.GeoDataFrame.from_features() takes STAC item dicts directly (item.to_dict())
#   - set crs="EPSG:4326" — STAC geometries are always lon/lat
#
# missions = ...
# missions.shape, missions.head()


## 3. The field plots

`plot_centroids_derived.csv` is one row per plot with WGS84 lat/lon. These are centroids,
not boundaries — good enough to find which mission a plot sits inside, not good enough to
clip anything to.

In [ ]:
# TODO(human): load the plot centroids as a GeoDataFrame.
#
#   - pd.read_csv, then gpd.points_from_xy(...)  — mind the argument order, it's (x, y)
#   - keep plot_id as a string; leading zeros matter ("0068", not 68)
#
# plots = ...


In [ ]:
# TODO(human): spatially join plots to missions to list the overlaps.
#
#   - gpd.sjoin(plots, missions, predicate="within")
#   - both sides must be in the same CRS before the join
#   - one plot can fall inside several missions, and one mission can hold several plots;
#     that's expected, don't drop the duplicates yet
#
# overlaps = ...
# how many distinct missions? how many distinct plots?


## 4. Cross-check against the complete footprint file

The STAC index is partial. Repeat the join against `all-mission-polygons-w-metadata.gpkg`
and compare the two candidate lists. Any plot that has a mission in the GeoPackage but not
in STAC is a mission whose processed products need checking by hand on CyVerse.

This is also where the `withheld_from_training` flag lives, if the layer carries it.

In [ ]:
# TODO(human): read the GeoPackage and redo the join.
#
#   - gpd.read_file(DRONE / "all-mission-polygons-w-metadata.gpkg")
#   - list its columns first; the metadata is the point of this file
#   - which plots gained a candidate mission that STAC didn't know about?


## 5. Rank the candidates

Task 1.2's four criteria, in the order that actually binds:

1. **Does the plot record species and live/dead status?** Nothing else matters if it doesn't — that's the ground truth. Currently unanswerable from public data; it's the question for OFO.
2. Does the mission have both an orthomosaic and a CHM? (All 299 STAC items do, so this only discriminates among the non-STAC missions.)
3. Can I see gray or red-brown dead crowns when I eyeball the orthomosaic?
4. Is the mission flagged `withheld_from_training`? Preferred, so the v2 test in Task 1.10 is genuinely out-of-sample.

Plot attributes for the ranking are in `ground-plot-catalog-datatable.html` — the widget JSON
stores them as column arrays, so parsing it is a small chore rather than a `read_html`.

In [ ]:
# TODO(human): pull the plot attribute table and join it onto the overlaps.
#
#   columns, in order: ID, area_ha, n_trees, basal_area, top_species, min_dbh_cm,
#                      min_ht_overhead_m, height_measured, year, project, license
#
# Then sort the candidates and write down, in a markdown cell, which site I'd pick and why.


## Site chosen

*(mission id, plot id, why this one, and what I'm giving up by choosing it)*